# EOH Run Monitor (Baseline vs Routed)

This notebook monitors logs produced by `notebooks/run_compare_baseline_routed.py`.
It reads:
- `runner_status.jsonl`
- `baseline/results/run_log.jsonl`
- `routed/results/run_log.jsonl`
- routed agent logs (`agent_observation/diagnosis/plan/critic`).

In [ ]:
from pathlib import Path
import json
import time

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import clear_output, display

plt.style.use('seaborn-v0_8-whitegrid')


In [ ]:
# Set this path if your output folder is elsewhere.
COMPARE_ROOT = Path('./compare_runs').resolve()

# Live refresh settings.
REFRESH_SECONDS = 8
MAX_LIVE_STEPS = 200  # stop automatically after this many refreshes

COMPARE_ROOT


In [ ]:
def read_jsonl(path: Path):
    if not path.exists():
        return []
    rows = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except Exception:
                pass
    return rows

def discover_seed_roots(compare_root: Path):
    seed_dirs = sorted([p for p in compare_root.glob('seed_*') if p.is_dir()])
    if seed_dirs:
        return seed_dirs
    return [compare_root]

def mode_result_dir(seed_root: Path, mode: str):
    candidate = seed_root / mode / 'results'
    if candidate.exists():
        return candidate
    # fallback for non-seeded single-run layouts
    flat = seed_root / 'results'
    if flat.exists() and mode == 'baseline':
        return flat
    return candidate

def load_run_log(compare_root: Path, mode: str):
    rows = []
    for seed_root in discover_seed_roots(compare_root):
        seed_name = seed_root.name if seed_root.name.startswith('seed_') else 'seed_single'
        path = mode_result_dir(seed_root, mode) / 'run_log.jsonl'
        for r in read_jsonl(path):
            r['seed'] = seed_name
            r['mode'] = mode
            rows.append(r)
    return pd.DataFrame(rows)

def load_agent_log(compare_root: Path, filename: str):
    rows = []
    for seed_root in discover_seed_roots(compare_root):
        seed_name = seed_root.name if seed_root.name.startswith('seed_') else 'seed_single'
        path = mode_result_dir(seed_root, 'routed') / filename
        for r in read_jsonl(path):
            r['seed'] = seed_name
            rows.append(r)
    return pd.DataFrame(rows)

def latest_status(compare_root: Path, n=12):
    status_path = compare_root / 'runner_status.jsonl'
    rows = read_jsonl(status_path)
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).tail(n)


In [ ]:
# Snapshot: latest runner status lines
status_df = latest_status(COMPARE_ROOT, n=15)
display(status_df)


In [ ]:
# Snapshot: run logs
base_df = load_run_log(COMPARE_ROOT, 'baseline')
routed_df = load_run_log(COMPARE_ROOT, 'routed')

print('baseline rows:', len(base_df))
print('routed rows:', len(routed_df))
display(base_df.tail(5))
display(routed_df.tail(5))


In [ ]:
def plot_progress(base_df: pd.DataFrame, routed_df: pd.DataFrame):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    for seed, sdf in base_df.groupby('seed'):
        y = sdf.get('train_fitness', sdf.get('best_fitness'))
        axes[0].plot(sdf['gen'], y, alpha=0.5, label=f'baseline:{seed}')
    for seed, sdf in routed_df.groupby('seed'):
        y = sdf.get('train_fitness', sdf.get('best_fitness'))
        axes[0].plot(sdf['gen'], y, alpha=0.8, linestyle='--', label=f'routed:{seed}')
    axes[0].set_title('Train Fitness vs Generation (lower is better)')
    axes[0].set_xlabel('Generation')
    axes[0].set_ylabel('Fitness')
    axes[0].legend(loc='best', fontsize=8)

    if not routed_df.empty:
        op_counts = routed_df['chosen_operator'].value_counts()
        axes[1].bar(op_counts.index.astype(str), op_counts.values)
    axes[1].set_title('Routed Operator Counts')
    axes[1].set_xlabel('Operator')
    axes[1].set_ylabel('Count')

    plt.tight_layout()
    plt.show()

plot_progress(base_df, routed_df)


In [ ]:
# Snapshot: recent agent decisions
diag_df = load_agent_log(COMPARE_ROOT, 'agent_diagnosis.jsonl')
plan_df = load_agent_log(COMPARE_ROOT, 'agent_plan.jsonl')
critic_df = load_agent_log(COMPARE_ROOT, 'agent_critic.jsonl')

print('diagnosis rows:', len(diag_df), '| plan rows:', len(plan_df), '| critic rows:', len(critic_df))
if not diag_df.empty:
    display(diag_df[['seed', 'gen', 'diagnosis_label', 'summary']].tail(10))
if not plan_df.empty:
    display(plan_df[['seed', 'gen', 'op_probs', 'parent_mix', 'prompt_modifiers']].tail(10))
if not critic_df.empty:
    display(critic_df[['seed', 'gen', 'verdict', 'reasons', 'chosen_operator']].tail(10))


In [ ]:
def live_monitor(compare_root: Path, refresh_s=8, max_steps=200):
    for step in range(max_steps):
        clear_output(wait=True)
        print(f'Live monitor step {step+1}/{max_steps} | root={compare_root}')

        status_df = latest_status(compare_root, n=10)
        if status_df.empty:
            print('No runner_status.jsonl records yet.')
        else:
            display(status_df[['time', 'message']].tail(10))

        base_df = load_run_log(compare_root, 'baseline')
        routed_df = load_run_log(compare_root, 'routed')
        print('baseline rows:', len(base_df), '| routed rows:', len(routed_df))
        if not base_df.empty or not routed_df.empty:
            plot_progress(base_df, routed_df)

        critic_df = load_agent_log(compare_root, 'agent_critic.jsonl')
        if not critic_df.empty:
            display(critic_df[['seed', 'gen', 'verdict', 'chosen_operator']].tail(8))

        time.sleep(refresh_s)

# Run and stop with KeyboardInterrupt when needed.
live_monitor(COMPARE_ROOT, refresh_s=REFRESH_SECONDS, max_steps=MAX_LIVE_STEPS)
